In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

windowed_df = pd.read_csv('../output/windown_sliced_data/windowed_df_10d.csv', parse_dates=['date', 'window_start', 'window_end', 'test_date'])
windowed_test = windowed_df.query('is_test_day == True').copy()

df_label_FI = pd.read_csv("../output/Anomaly_delirium_Revised/FI_anomaly_labels.csv")
df_label_LSTM = pd.read_csv("../output/Anomaly_delirium_Revised/LSTM_anomaly_labels.csv")

# shared keys
keys = ["patient_id", "test_date"]

# make sure the merge key has the same dtype in both dataframes
df_label_FI["test_date"] = pd.to_datetime(df_label_FI["test_date"])
windowed_test["test_date"] = pd.to_datetime(windowed_test["test_date"])

# only keep selected columns from windowed_test
B_sub = windowed_test[
    keys + [
        "entropy_rate",
        "early_warning_score",
        "sleep_quality_score",
        "agitation_counts",
        "uti_happen"
    ]
    ]

# merge
FI_visual_df = df_label_FI.merge(B_sub, on=keys, how="left")

FI_visual_df

,patient_id,window_start,test_date,anomaly_score,anomaly_label,entropy_rate,early_warning_score,sleep_quality_score,agitation_counts,uti_happen
0,16f4b,2019-04-28,2019-05-08,0.085888,1,0.621309,0.0,0.750000,0.0,0.0
1,16f4b,2019-05-03,2019-05-13,0.100597,1,0.623305,0.0,1.500000,0.0,0.0
2,1fbe4,2019-04-25,2019-05-05,0.079158,1,0.681941,1.0,0.800000,0.0,0.0
3,1fbe4,2019-04-26,2019-05-06,0.056187,1,0.668119,0.0,0.000000,0.0,0.0
4,1fbe4,2019-04-27,2019-05-07,0.008614,1,0.681583,1.0,1.250000,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...
487,ec812,2019-05-29,2019-06-08,0.067374,1,0.649826,0.0,1.285714,0.0,0.0
488,ec812,2019-06-12,2019-06-22,-0.011182,-1,0.623560,0.0,0.666667,0.0,0.0
489,ec812,2019-06-13,2019-06-23,0.046199,1,0.708682,0.0,0.875000,0.0,0.0
490,ec812,2019-06-14,2019-06-24,0.119338,1,0.656330,0.0,0.888889,0.0,0.0


In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# merge FI and LSTM results
FI_visual_df['test_date'] = pd.to_datetime(FI_visual_df['test_date'])
df_label_LSTM['test_date'] = pd.to_datetime(df_label_LSTM['test_date'])

merged = FI_visual_df[['patient_id', 'test_date', 'anomaly_label',
                       'entropy_rate', 'early_warning_score',
                       'sleep_quality_score', 'agitation_counts', 'uti_happen']].merge(
    df_label_LSTM[['patient_id', 'test_date', 'anomaly_label']],
    on=['patient_id', 'test_date'],
    suffixes=('_fi', '_lstm')
)

for id_select in merged['patient_id'].unique():
    df_person = merged[merged['patient_id'] == id_select].copy()
    df_person = df_person.sort_values('test_date')

    # ── classify each date by detection source ───────────────────────────────
    fi_anomaly = set(df_person[df_person['anomaly_label_fi'] == -1]['test_date']
                     .dt.strftime('%m-%d').values)
    lstm_anomaly = set(df_person[df_person['anomaly_label_lstm'] == -1]['test_date']
                       .dt.strftime('%m-%d').values)
    shared_anomaly = fi_anomaly & lstm_anomaly
    fi_only = fi_anomaly - shared_anomaly
    lstm_only = lstm_anomaly - shared_anomaly

    data_plot = df_person[['test_date', 'entropy_rate', 'early_warning_score',
                           'sleep_quality_score', 'agitation_counts', 'uti_happen']].copy()
    data_plot.rename(columns={
        "entropy_rate": "entropy rate",
        "early_warning_score": "early warning score",
        "sleep_quality_score": "sleep quality score",
        "agitation_counts": "agitation counts",
        "uti_happen": "uti counts"
    }, inplace=True)
    data_plot['test_date'] = data_plot['test_date'].dt.strftime('%m-%d').astype(str)
    data_plot.set_index('test_date', inplace=True)
    x_labels = data_plot.index.tolist()

    plt.figure(figsize=(12, 10))

    for i, column in enumerate(data_plot.columns):
        ax = plt.subplot(5, 1, i + 1)
        ax.plot(x_labels, data_plot[column], color='steelblue', linewidth=1.5)

        # ── FI only: red ─────────────────────────────────────────────────────
        fi_dates = [d for d in fi_only if d in x_labels]
        if fi_dates:
            ax.scatter(fi_dates, data_plot[column][fi_dates],
                       color='red', zorder=5, s=40, label='IF only' if i == 0 else '')

        # ── LSTM only: blue ───────────────────────────────────────────────────
        lstm_dates = [d for d in lstm_only if d in x_labels]
        if lstm_dates:
            ax.scatter(lstm_dates, data_plot[column][lstm_dates],
                       color='blue', zorder=5, s=40, label='LSTM only' if i == 0 else '')

        # ── shared: green ─────────────────────────────────────────────────────
        shared_dates = [d for d in shared_anomaly if d in x_labels]
        if shared_dates:
            ax.scatter(shared_dates, data_plot[column][shared_dates],
                       color='green', zorder=5, s=40, label='Shared' if i == 0 else '')

        ax.set_title(f'{column}')
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.set_xticks(range(len(x_labels)))
        ax.set_xticklabels(x_labels, rotation=90)
        ax.set_ylim(data_plot[column].min() - 1, data_plot[column].max() + 1)

    # ── single legend on first subplot ───────────────────────────────────────
    axes = plt.gcf().get_axes()
    axes[0].legend(loc='upper right', fontsize=8)

    plt.suptitle(f'Participant {id_select} — IF vs LSTM Anomaly Detection',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../output/Anomaly_delirium_Revised/combined_visual/anomaly_combined_' + id_select + '.png',
                dpi=300)
    plt.close()

C:\Users\Sola\AppData\Local\Temp\ipykernel_78468\1968844136.py:76: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  axes[0].legend(loc='upper right', fontsize=8)


In [3]:
import pandas as pd
import numpy as np

# merge FI and LSTM results
FI_visual_df['test_date'] = pd.to_datetime(FI_visual_df['test_date'])
df_label_LSTM['test_date'] = pd.to_datetime(df_label_LSTM['test_date'])

merged = FI_visual_df[['patient_id', 'test_date', 'anomaly_label']].merge(
    df_label_LSTM[['patient_id', 'test_date', 'anomaly_label']],
    on=['patient_id', 'test_date'],
    suffixes=('_fi', '_lstm')
)

def count_clusters(dates, min_days=2):
    """Count consecutive clusters of at least min_days length."""
    if len(dates) == 0:
        return 0
    dates = sorted(pd.to_datetime(list(dates)))
    clusters = 0
    cluster_len = 1
    for i in range(1, len(dates)):
        if (dates[i] - dates[i-1]).days == 1:
            cluster_len += 1
        else:
            if cluster_len >= min_days:
                clusters += 1
            cluster_len = 1
    if cluster_len >= min_days:
        clusters += 1
    return clusters

summary_rows = []

for pid, group in merged.groupby('patient_id'):
    group = group.sort_values('test_date')

    fi_dates = set(group[group['anomaly_label_fi'] == -1]['test_date'].tolist())
    lstm_dates = set(group[group['anomaly_label_lstm'] == -1]['test_date'].tolist())
    shared_dates = fi_dates & lstm_dates
    fi_only_dates = fi_dates - shared_dates
    lstm_only_dates = lstm_dates - shared_dates
    all_flagged = fi_dates | lstm_dates

    summary_rows.append({
        'patient_id': pid,
        'total_days': len(group),
        'fi_anomaly_days': len(fi_dates),
        'lstm_anomaly_days': len(lstm_dates),
        'shared_days': len(shared_dates),
        'fi_only_days': len(fi_only_dates),
        'lstm_only_days': len(lstm_only_dates),
        'agreement_rate': round(len(shared_dates) / len(all_flagged), 3)
        if len(all_flagged) > 0 else 0,
        'fi_clusters': count_clusters(fi_dates),
        'lstm_clusters': count_clusters(lstm_dates),
        'shared_clusters': count_clusters(shared_dates),
    })

summary_df = pd.DataFrame(summary_rows)

# ── add totals row ────────────────────────────────────────────────────────────
totals = summary_df.drop(columns='patient_id').sum().to_dict()
totals['agreement_rate'] = round(summary_df['agreement_rate'].mean(), 3)
totals['patient_id'] = 'TOTAL/MEAN'
summary_df = pd.concat(
    [summary_df, pd.DataFrame([totals])], ignore_index=True
)

print(summary_df.to_string(index=False))
summary_df.to_csv(
    '../output/Anomaly_delirium_Revised/algorithm_comparison_summary.csv',
    index=False
)

patient_id  total_days  fi_anomaly_days  lstm_anomaly_days  shared_days  fi_only_days  lstm_only_days  agreement_rate  fi_clusters  lstm_clusters  shared_clusters
     16f4b         2.0              0.0                1.0          0.0           0.0             1.0           0.000          0.0            0.0              0.0
     1fbe4        39.0             12.0                6.0          4.0           8.0             2.0           0.286          4.0            1.0              1.0
     30a32        52.0             13.0                8.0          5.0           8.0             3.0           0.312          3.0            1.0              1.0
     55cd4        50.0             13.0                8.0          6.0           7.0             2.0           0.400          1.0            2.0              1.0
     93c14        27.0              3.0                4.0          1.0           2.0             3.0           0.167          0.0            0.0              0.0
     96adf        39.0

In [4]:
# compute per-participant fluctuation for each predictor
# and correlate with anomaly event counts
PREDICTORS = ['entropy_rate', 'early_warning_score',
              'sleep_quality_score', 'agitation_counts', 'uti_happen']
merged = FI_visual_df[['patient_id', 'test_date', 'anomaly_label',
                       'entropy_rate', 'early_warning_score',
                       'sleep_quality_score', 'agitation_counts', 'uti_happen']].merge(
    df_label_LSTM[['patient_id', 'test_date', 'anomaly_label']],
    on=['patient_id', 'test_date'],
    suffixes=('_fi', '_lstm')
)

from scipy.stats import spearmanr  # use Spearman as counts are not normally distributed
from scipy.stats import pearsonr
from scipy.stats import spearmanr

fluctuation_rows = []

for pid, group in merged.groupby('patient_id'):
    group = group.sort_values('test_date')

    # fluctuation per predictor, take the maximum across all predictors
    fluct_per_predictor = []
    for col in PREDICTORS:
        series = group[col].dropna().values
        if len(series) > 1:
            fluct_per_predictor.append(np.mean(np.abs(np.diff(series))))

    fluctuation_rows.append({
        'patient_id': pid,
        'fluct_max': max(fluct_per_predictor) if fluct_per_predictor else np.nan,
        'fi_anomaly_days': (group['anomaly_label_fi'] == -1).sum(),
        'lstm_anomaly_days': (group['anomaly_label_lstm'] == -1).sum()
    })

fluct_df = pd.DataFrame(fluctuation_rows)
fluct_df = fluct_df.fillna(0).copy()
fi_corr_s, fi_p_s = spearmanr(fluct_df['fluct_max'], fluct_df['fi_anomaly_days'])
lstm_corr_s, lstm_p_s = spearmanr(fluct_df['fluct_max'], fluct_df['lstm_anomaly_days'])
fi_corr, fi_p = pearsonr(fluct_df['fluct_max'], fluct_df['fi_anomaly_days'])
lstm_corr, lstm_p = pearsonr(fluct_df['fluct_max'], fluct_df['lstm_anomaly_days'])

print(f"IF   — Spearman r = {fi_corr_s:.3f}, p = {fi_p_s:.3f}")
print(f"LSTM — Spearman r = {lstm_corr_s:.3f}, p = {lstm_p_s:.3f}")
print(f"IF   — Pearson r = {fi_corr:.3f}, p = {fi_p:.3f}")
print(f"LSTM — Pearson r = {lstm_corr:.3f}, p = {lstm_p:.3f}")


fluct_df.to_csv(
    '../output/Anomaly_delirium_Revised/algorithm_comparison_fluctuation_correlation.csv',
    index=False
)

IF   — Spearman r = 0.304, p = 0.313
LSTM — Spearman r = -0.130, p = 0.672
IF   — Pearson r = 0.221, p = 0.468
LSTM — Pearson r = -0.055, p = 0.859
